[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/41_agent_loop_solution.ipynb)

#  Solution: ReAct Agent Loop

Reference solution for the ReAct agent loop implementation.

```text
1. Send query to model_fn
2. Parse response for Final Answer or Action
3. If Action found, execute tool and feed Observation back
4. Repeat until Final Answer or max_steps exceeded
```

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch
import re

In [ ]:
#  SOLUTION

def react_agent_loop(model_fn, tools, query, max_steps=5):
    """Minimal ReAct agent loop.
    
    Reference: Yao et al. "ReAct: Synergizing Reasoning and Acting in Language Models"
    Pattern: Thought  Action  Observation loop, common in LangChain / AutoGPT-style agents.
    """
    prompt = query
    
    for step in range(max_steps):
        response = model_fn(prompt)
        
        # 1) Check for Final Answer first
        fa = re.search(r'Final Answer:\s*(.+?)$', response, re.MULTILINE | re.DOTALL)
        if fa:
            return fa.group(1).strip()
        
        # 2) Try to extract Action
        act = re.search(r'Action:\s*(\w+)', response)
        if act and act.group(1) in tools:
            tool_name = act.group(1)
            inp = re.search(r'Action Input:\s*(.+?)$', response, re.MULTILINE | re.DOTALL)
            tool_arg = inp.group(1).strip() if inp else ""
            result = tools[tool_name](tool_arg)
            prompt += f"\nObservation: {result}"
    
    return None

In [ ]:
#  Verify
responses = iter([
    "Thought: Let me look this up.\nAction: search\nAction Input: capital of France",
    "Thought: Got it.\nFinal Answer: Paris",
])
mock_model = lambda prompt: next(responses)
tools = {"search": lambda q: f"Paris is capital of France"}
res = react_agent_loop(mock_model, tools, "capital of France?")
print("Result:", res)

In [ ]:
# Run judge
from torch_judge import check
check("agent_loop")